<a href="https://colab.research.google.com/github/DurabH/CodeAlpha_Credit_Scoring_Model/blob/main/CreditScoringModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Step 1: Mount Google Drive
from google.colab import drive
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

drive.mount('/content/drive')

# Step 2: Define the file path and load the data
file_path = '/content/drive/MyDrive/CodeAlpha/german_credit_data.csv'

if os.path.exists(file_path):
    df = pd.read_csv(file_path, index_col=0)
    print("Dataset loaded successfully!")
    print(f"Dataset Shape: {df.shape}\n")
else:
    print(f"Error: File not found at {file_path}. Please check your Google Drive folder structure.")

# --- PREPROCESSING PIPELINE ---

# Step 3: Handle Missing Values
df['Saving accounts'] = df['Saving accounts'].fillna('unknown')
df['Checking account'] = df['Checking account'].fillna('unknown')

# *** NEW CRUCIAL STEP: Separate Features (X) from the Real Target (y) ***
X = df.drop(columns=['Risk'])
y = df['Risk'].map({'good': 0, 'bad': 1}) # Map ground truth text labels to numbers

# Step 4: Identify Feature Types
numerical_features = ['Age', 'Credit amount', 'Duration']
categorical_features = ['Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']

print("Initial Features Preview:")
print(X.head(), "\n")

# Step 5: Build the Transformer Pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

# Step 6: Transform ONLY the features (X), not the whole dataframe
X_processed = preprocessor.fit_transform(X)

# Convert back to a DataFrame to easily visualize the processed data
encoded_cat_features = preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_features)
all_features = numerical_features + list(encoded_cat_features)

processed_df = pd.DataFrame(X_processed, columns=all_features)

print("--- Preprocessing Complete ---")
print(f"Processed Dataset Shape: {processed_df.shape}")
print("\nProcessed Data Preview (First 5 Rows):")
print(processed_df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset loaded successfully!
Dataset Shape: (1000, 10)

Initial Features Preview:
   Age     Sex  Job Housing Saving accounts Checking account  Credit amount  \
0   67    male    2     own         unknown           little           1169   
1   22  female    2     own          little         moderate           5951   
2   49    male    1     own          little          unknown           2096   
3   45    male    2    free          little           little           7882   
4   53    male    2    free          little           little           4870   

   Duration              Purpose  
0         6             radio/TV  
1        48             radio/TV  
2        12            education  
3        42  furniture/equipment  
4        24                  car   

--- Preprocessing Complete ---
Processed Dataset Shape: (1000, 29)

Processed Data Preview (First 5 Ro

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

# --- STEP 1: SPLIT THE DATA ---
# Using the X_processed and y from your previous step
# Stratify=y maintains a balanced ratio of good vs. bad credit in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Features Shape: {X_train.shape}")
print(f"Testing Features Shape: {X_test.shape}\n")

# --- STEP 2: INITIALIZE THE MODELS ---
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=5),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100)
}

# --- STEP 3: TRAIN AND EVALUATE EACH MODEL ---
for model_name, model in models.items():
    print("="*60)
    print(f" Training and Evaluating: {model_name} ")
    print("="*60)

    # Train the model
    model.fit(X_train, y_train)

    # Generate predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] # Probability estimates for ROC-AUC

    # Output metrics
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["Good Credit (0)", "Bad Credit (1)"]))

    auc_score = roc_auc_score(y_test, y_pred_proba)
    print(f"ROC-AUC Score: {auc_score:.4f}\n")

    cm = confusion_matrix(y_test, y_pred)
    print("Confusion Matrix Layout:")
    print(f"[ [ True Negatives (Correct Good),   False Positives (Missed Bad) ]")
    print(f"  [ False Negatives (Missed Good), True Positives (Correct Bad) ] ]")
    print("\nActual Matrix Values:")
    print(cm)
    print("\n")

Training Features Shape: (800, 29)
Testing Features Shape: (200, 29)

 Training and Evaluating: Logistic Regression 

Classification Report:
                 precision    recall  f1-score   support

Good Credit (0)       0.77      0.89      0.82       140
 Bad Credit (1)       0.59      0.38      0.46        60

       accuracy                           0.73       200
      macro avg       0.68      0.63      0.64       200
   weighted avg       0.72      0.73      0.72       200

ROC-AUC Score: 0.7613

Confusion Matrix Layout:
[ [ True Negatives (Correct Good),   False Positives (Missed Bad) ]
  [ False Negatives (Missed Good), True Positives (Correct Bad) ] ]

Actual Matrix Values:
[[124  16]
 [ 37  23]]


 Training and Evaluating: Decision Tree 

Classification Report:
                 precision    recall  f1-score   support

Good Credit (0)       0.79      0.74      0.76       140
 Bad Credit (1)       0.47      0.53      0.50        60

       accuracy                           0.

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import os

# 1. Load Data
file_path = '/content/drive/MyDrive/CodeAlpha/german_credit_data.csv'
df = pd.read_csv(file_path, index_col=0)

df['Saving accounts'] = df['Saving accounts'].fillna('unknown')
df['Checking account'] = df['Checking account'].fillna('unknown')

# --- ADVANCED FEATURE ENGINEERING ---
# Create a proxy feature for monthly payment pressure
df['Monthly_Burden'] = df['Credit amount'] / df['Duration']
# Create a proxy feature for credit risk scaling by age
df['Credit_to_Age_Ratio'] = df['Credit amount'] / df['Age']

# Separate Features and Target
X = df.drop(columns=['Risk'])
y = df['Risk'].map({'good': 0, 'bad': 1})

# Define updated features list
numerical_features = ['Age', 'Credit amount', 'Duration', 'Monthly_Burden', 'Credit_to_Age_Ratio']
categorical_features = ['Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']

# 2. Pipeline Processing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

X_processed = preprocessor.fit_transform(X)

# 3. Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

# --- ADDRESS CLASS IMBALANCE WITH SMOTE ---
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# --- HYPERPARAMETER TUNING VIA GRID SEARCH ---
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [8, 12, 15],
    'min_samples_split': [2, 5, 10],
    'class_weight': ['balanced', None]
}

rf_base = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5, scoring='roc_auc', n_jobs=-1)

print("Tuning hyperparameters and training optimized model (this may take a few seconds)...")
grid_search.fit(X_train_balanced, y_train_balanced)

# 4. Evaluation of Optimized Model
best_rf = grid_search.best_estimator_
y_pred = best_rf.predict(X_test)
y_pred_proba = best_rf.predict_proba(X_test)[:, 1]

print("\n" + "="*50)
print(" OPTIMIZED RANDOM FOREST RESULTS ")
print("="*50)
print(f"Best Parameters Found: {grid_search.best_params_}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Good Credit (0)", "Bad Credit (1)"]))
print(f"Optimized ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Tuning hyperparameters and training optimized model (this may take a few seconds)...

 OPTIMIZED RANDOM FOREST RESULTS 
Best Parameters Found: {'class_weight': 'balanced', 'max_depth': 15, 'min_samples_split': 2, 'n_estimators': 300}

Classification Report:
                 precision    recall  f1-score   support

Good Credit (0)       0.82      0.84      0.83       140
 Bad Credit (1)       0.61      0.57      0.59        60

       accuracy                           0.76       200
      macro avg       0.71      0.70      0.71       200
   weighted avg       0.76      0.76      0.76       200

Optimized ROC-AUC Score: 0.7763

Confusion Matrix:
[[118  22]
 [ 26  34]]


In [7]:
#Fully optimized and finely tuned model code

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.over_sampling import SMOTE

# 1. Reload and Preprocess Data
file_path = '/content/drive/MyDrive/CodeAlpha/german_credit_data.csv'
df = pd.read_csv(file_path, index_col=0)

df['Saving accounts'] = df['Saving accounts'].fillna('unknown')
df['Checking account'] = df['Checking account'].fillna('unknown')

# Feature Engineering
df['Monthly_Burden'] = df['Credit amount'] / df['Duration']
df['Credit_to_Age_Ratio'] = df['Credit amount'] / df['Age']

X = df.drop(columns=['Risk'])
y = df['Risk'].map({'good': 0, 'bad': 1})

numerical_features = ['Age', 'Credit amount', 'Duration', 'Monthly_Burden', 'Credit_to_Age_Ratio']
categorical_features = ['Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
    ]
)

X_processed = preprocessor.fit_transform(X)

# 2. Train-Test Split & SMOTE Balancing
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

# --- EXPANDED ADVANCED HYPERPARAMETER TUNING ---
fine_tuned_grid = {
    'n_estimators': [300, 400, 500],          # Increased forest sizes
    'max_depth': [12, 15, 18, None],          # Deeper bounds
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2, 4],            # Controls leaf node size to stop overfitting
    'max_features': ['sqrt', 'log2'],         # Restricts features per tree split
    'criterion': ['gini', 'entropy'],         # Tests different impurity math metrics
    'class_weight': ['balanced']
}

rf_base = RandomForestClassifier(random_state=42)

# Using scoring='f1_macro' to explicitly force the search engine to optimize
# for BOTH good and bad credit equally, rather than over-focusing on overall accuracy.
grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=fine_tuned_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

print("Running deep structural fine-tuning sweep (this might take 15-30 seconds)...")
grid_search.fit(X_train_balanced, y_train_balanced)

# 3. Final Optimized Evaluation
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_pred_proba = best_model.predict_proba(X_test)[:, 1]

print("\n" + "="*50)
print(" FINAL DEEP TUNED MODEL RESULTS ")
print("="*50)
print(f"Optimal Hyperparameters: {grid_search.best_params_}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Good Credit (0)", "Bad Credit (1)"]))
print(f"Final ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Running deep structural fine-tuning sweep (this might take 15-30 seconds)...

 FINAL DEEP TUNED MODEL RESULTS 
Optimal Hyperparameters: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 18, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}

Classification Report:
                 precision    recall  f1-score   support

Good Credit (0)       0.81      0.82      0.82       140
 Bad Credit (1)       0.57      0.55      0.56        60

       accuracy                           0.74       200
      macro avg       0.69      0.69      0.69       200
   weighted avg       0.74      0.74      0.74       200

Final ROC-AUC Score: 0.7814

Confusion Matrix:
[[115  25]
 [ 27  33]]


In [8]:
#@title 🚀 Interactive Credit Risk Predictor (Test Your Model)
#@markdown Enter an applicant's details below to see if the final tuned model approves or denies their credit.

import ipywidgets as widgets
from IPython.display import display, clear_output

# Create input fields matching the original dataset
Age = widgets.IntSlider(value=35, min=18, max=100, step=1, description='Age:')
Credit_amount = widgets.IntSlider(value=3000, min=100, max=20000, step=100, description='Credit Amt:')
Duration = widgets.IntSlider(value=24, min=4, max=72, step=1, description='Duration(Mo):')

Sex = widgets.Dropdown(options=['male', 'female'], value='male', description='Sex:')
Job = widgets.Dropdown(options=[0, 1, 2, 3], value=2, description='Job Level:')
Housing = widgets.Dropdown(options=['own', 'free', 'rent'], value='own', description='Housing:')
Saving_accounts = widgets.Dropdown(options=['little', 'moderate', 'quite rich', 'rich', 'unknown'], value='little', description='Savings:')
Checking_account = widgets.Dropdown(options=['little', 'moderate', 'rich', 'unknown'], value='little', description='Checking:')
Purpose = widgets.Dropdown(options=['radio/TV', 'education', 'furniture/equipment', 'car', 'business', 'domestic appliances', 'repairs', 'vacation/others'], value='car', description='Purpose:')

button = widgets.Button(description="Predict Credit Score", button_style='success')
output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()

        # 1. Collect inputs into a single-row DataFrame
        input_data = pd.DataFrame([{
            'Age': Age.value,
            'Sex': Sex.value,
            'Job': Job.value,
            'Housing': Housing.value,
            'Saving accounts': Saving_accounts.value,
            'Checking account': Checking_account.value,
            'Credit amount': Credit_amount.value,
            'Duration': Duration.value,
            'Purpose': Purpose.value
        }])

        # 2. Apply the same manual feature engineering ratios
        input_data['Monthly_Burden'] = input_data['Credit amount'] / input_data['Duration']
        input_data['Credit_to_Age_Ratio'] = input_data['Credit amount'] / input_data['Age']

        # 3. Process features through the pipeline preprocessor
        processed_input = preprocessor.transform(input_data)

        # 4. Predict using the final deep-tuned model
        prediction = best_model.predict(processed_input)[0]
        probability = best_model.predict_proba(processed_input)[0][1]

        # 5. Display stylized results
        print("="*40)
        print("         CREDIT ASSESSMENT SYSTEM         ")
        print("="*40)
        if prediction == 0:
            print(f"🎉 STATUS: APPROVED (Low Risk)")
            print(f"Confidence Score: {((1 - probability) * 100):.2f}% chance of reliable repayment.")
        else:
            print(f"❌ STATUS: REJECTED (High Risk)")
            print(f"Risk Score: {(probability * 100):.2f}% chance of credit default.")
        print("="*40)

button.on_click(on_button_clicked)

# Display layout
display(widgets.VBox([Age, Credit_amount, Duration, Sex, Job, Housing, Saving_accounts, Checking_account, Purpose, button, output]))